# Base de Datos SQL.

Aqui consideraremos un apartado secundario. Donde construiremos una base de datos con SQLite para compactar la información requerida para el analisis. 

## Creación de la base de datos.

Comenzaremos creando un esquema en blanco. 

In [49]:
from scripts.config import ROOT
import duckdb

conn = duckdb.connect(ROOT / "models" / "agro.duckdb")
conn.execute("INSTALL spatial")
conn.execute("LOAD spatial")

## Indice.

Para esta base de datos usaremos los municipios como clave primaria. Para ello primero debemos obtener la clave geografica municipal (CVEGEO). Esto la obtendremos a partir del Marco Geoestádistico.

## Producción Agricola.

A partir de la información precargada podemos obtener 

In [50]:
from scripts.config import PRELOADS, DATA_DIR
import os
from zipfile import ZipFile
import geopandas as gpd

with ZipFile(PRELOADS['MAPA GENERAL'], "r") as mexico:
    os.makedirs(DATA_DIR / "raw" / "INEGI", exist_ok=True)
    mexico.extractall(DATA_DIR / "raw" / "INEGI")

maps_folder = DATA_DIR / "raw" / "INEGI" / "Mexico" / "conjunto_de_datos"
maps = [file for file in os.listdir(maps_folder) if file.endswith(".shp")]
print(maps)


['00ent.shp', '00mun.shp']


Usando el conjunto de datos de los municipios crearemos la primera tabla con la clave CVEGEO como clave primaria. 

In [51]:
mexico = gpd.read_file(maps_folder / maps[1])
mexico

,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO,geometry
0,01001,01,001,Aguascalientes,"POLYGON ((2488980.227 1117358.447, 2489013.495..."
1,01008,01,008,San José de Gracia,"POLYGON ((2453326.553 1143576.423, 2453363.325..."
2,01011,01,011,San Francisco de los Romo,"POLYGON ((2483638.349 1120034.022, 2483946.175..."
3,01009,01,009,Tepezalá,"POLYGON ((2481826.447 1149055.253, 2481589.222..."
4,01007,01,007,Rincón de Romos,"POLYGON ((2476760.231 1150329.046, 2476783.634..."
...,...,...,...,...,...
2473,32004,32,004,Benito Juárez,"POLYGON ((2336381.078 1064884.55, 2336405.02 1..."
2474,32043,32,043,Susticacán,"POLYGON ((2380699.092 1186256.441, 2380731.393..."
2475,32042,32,042,Sombrerete,"POLYGON ((2323463.845 1340295.088, 2323485.623..."
2476,32014,32,014,General Francisco R. Murguía,"POLYGON ((2395056.891 1381032.418, 2395060.211..."


Formateamos el conjunto de datos.

In [52]:
mexico_sql = mexico.set_index('CVEGEO').sort_index()
mexico_sql

,CVE_ENT,CVE_MUN,NOMGEO,geometry
CVEGEO,,,,
01001,01,001,Aguascalientes,"POLYGON ((2488980.227 1117358.447, 2489013.495..."
01002,01,002,Asientos,"POLYGON ((2493552.647 1142361.087, 2493586.6 1..."
01003,01,003,Calvillo,"POLYGON ((2429646.495 1120918.059, 2429711.746..."
01004,01,004,Cosío,"POLYGON ((2469550.852 1159302.643, 2469738.209..."
01005,01,005,Jesús María,"POLYGON ((2463657.764 1116053.82, 2463738.92 1..."
...,...,...,...,...
32054,32,054,Villa Hidalgo,"POLYGON ((2531125.953 1166440.462, 2531160.381..."
32055,32,055,Villanueva,"POLYGON ((2419745.763 1183719.974, 2419854.708..."
32056,32,056,Zacatecas,"POLYGON ((2421780.086 1201989.77, 2421852.949 ..."


In [53]:
gdf_wkb = mexico.copy()
gdf_wkb["geom_wkb"] = gdf_wkb.geometry.to_wkb()
gdf_wkb = gdf_wkb.drop(columns="geometry")
conn.register("tmp", gdf_wkb)
conn.execute("""
    CREATE or REPLACE TABLE municipios AS
    SELECT *, ST_GeomFromWKB(geom_wkb) AS geom
    FROM tmp
""")

In [54]:
conn.close()

In [55]:
conn = duckdb.connect(ROOT / "models" / "agro.duckdb")
conn.execute("INSTALL spatial")
conn.execute("LOAD spatial")
df = conn.sql("SELECT *, ST_AsWKB(geom) AS geom_wkb FROM municipios").df()

mun = gpd.GeoDataFrame(
    df,
    geometry=gpd.GeoSeries.from_wkb(df["geom_wkb"].map(bytes)),
)
mun = mun.set_crs(mexico.crs)
mun

,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO,geom_wkb,geom,geom_wkb_1,geometry
0,01001,01,001,Aguascalientes,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 161, 8, 0, 0, 193,...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 161, 8, 0, 0, 193,...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 161, 8, 0, 0, 193,...","POLYGON ((2488980.227 1117358.447, 2489013.495..."
1,01008,01,008,San José de Gracia,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 94, 10...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 94, 10...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 94, 10...","POLYGON ((2453326.553 1143576.423, 2453363.325..."
2,01011,01,011,San Francisco de los Romo,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 94, 4, 0, 0, 86, 1...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 94, 4, 0, 0, 86, 1...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 94, 4, 0, 0, 86, 1...","POLYGON ((2483638.349 1120034.022, 2483946.175..."
3,01009,01,009,Tepezalá,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 61, 3, 0, 0, 76, 4...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 61, 3, 0, 0, 76, 4...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 61, 3, 0, 0, 76, 4...","POLYGON ((2481826.447 1149055.253, 2481589.222..."
4,01007,01,007,Rincón de Romos,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 20, 4, 0, 0, 45, 2...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 20, 4, 0, 0, 45, 2...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 20, 4, 0, 0, 45, 2...","POLYGON ((2476760.231 1150329.046, 2476783.634..."
...,...,...,...,...,...,...,...,...
2473,32004,32,004,Benito Juárez,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 205, 4, 0, 0, 214,...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 205, 4, 0, 0, 214,...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 205, 4, 0, 0, 214,...","POLYGON ((2336381.078 1064884.55, 2336405.02 1..."
2474,32043,32,043,Susticacán,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 194, 4, 0, 0, 51, ...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 194, 4, 0, 0, 51, ...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 194, 4, 0, 0, 51, ...","POLYGON ((2380699.092 1186256.441, 2380731.393..."
2475,32042,32,042,Sombrerete,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 82, 20, 0, 0, 195,...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 82, 20, 0, 0, 195,...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 82, 20, 0, 0, 195,...","POLYGON ((2323463.845 1340295.088, 2323485.623..."
2476,32014,32,014,General Francisco R. Murguía,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 187, 11, 0, 0, 188...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 187, 11, 0, 0, 188...","[1, 3, 0, 0, 0, 1, 0, 0, 0, 187, 11, 0, 0, 188...","POLYGON ((2395056.891 1381032.418, 2395060.211..."
